# 🎯 Self-Querying Retrieval (SQR)

A self-querying retriever answers questions that plain vector search **cannot**, such as
*"a book rated above 4.5 published after 2003."* Similarity search compares meaning; it has no
notion of *greater than*. Ratings and years live in **metadata**, not in the embedded text.

Self-querying closes that gap by putting an LLM in front of the retriever. The LLM splits your
question into two parts:

| Part | Example | Handled by |
|---|---|---|
| **Semantic query** | `"deep themes"` | Vector similarity |
| **Structured filter** | `rating > 4.5` | The vector store's metadata filter |

## Learning Objectives
1. **The metadata gap** — see plain similarity search fail on a numeric constraint
2. **Query construction** — watch an LLM split one question into a query plus a filter
3. **`AttributeInfo` schemas** — describe your metadata so the LLM knows what it may filter on
4. **Translators** — convert the abstract filter into store-specific syntax (e.g. Chroma's `$gt`)
5. **`enable_limit`** — let the question itself control how many results come back

## Prerequisites
- A `.env` at the repo root with `OPENAI_API_KEY` and `EXPERIENTIALLABS_API_KEY`
- Packages: `langchain-chroma`, `langchain-classic`, `langchain-community`, `langchain-openai`
- Familiarity with embeddings and vector stores (see `01_Introduction_to_RAG/`)

---
## 🧠 Part 1: Why Plain Retrieval Is Not Enough

Dense retrieval embeds your question and returns the nearest chunks. That works well for
*"books about identity and belonging"* — a purely semantic request.

It breaks down the moment a question contains a **constraint on a fact**:

- *"...rated **above 4.5**"* → a numeric comparison
- *"...published **between 2003 and 2015**"* → a range
- *"...from the **USA or UK**"* → a set membership

Embeddings cannot express any of these. The string `"rating above 4.5"` has a vector, but that
vector encodes the *phrase*, not the *arithmetic*. A book rated 4.3 is just as "semantically close"
to it as one rated 4.9.

### Key Concepts:
- **Semantic query**: the part of the question answered by meaning — matched via embeddings.
- **Structured filter**: the part answered by facts — applied as a metadata `WHERE` clause.
- **Query constructor**: the LLM chain that separates the two.
- **Translator**: converts the store-agnostic filter into a specific store's syntax.

> **Key Insight**: self-querying does not make retrieval smarter. It routes each half of the
> question to the tool that can actually answer it — embeddings for meaning, the database for facts.

---
## ⚙️ Part 2: Environment Setup

### 2.1 Imports

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
import os
import warnings

from dotenv import load_dotenv

# LangChain core
from langchain_core.documents import Document
from langchain_core.structured_query import Comparison

# Integrations
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Self-querying machinery
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator

# Project helper (LLM factory)
from helpers import get_experientiallabs_llm

warnings.filterwarnings("ignore")

print("✅ Imports loaded successfully!")

### 2.2 Credentials and LangSmith Tracing

> **Note**: use `LANGSMITH_PROJECT`, not the legacy `LANGCHAIN_PROJECT`. The SDK checks the
> `LANGSMITH_` prefix **first**, so a `LANGSMITH_PROJECT` already set in `.env` would silently
> win and these traces would land in that project instead of this one.

In [ ]:
# ============================================================================
# CONFIGURATION: Credentials and tracing
# ============================================================================
load_dotenv()

os.environ["LANGSMITH_PROJECT"] = "Self-Querying-Retrieval"

print(f"✅ OpenAI key present:  {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"✅ LangSmith tracing:   {os.getenv('LANGSMITH_TRACING')}")
print(f"✅ LangSmith project:   {os.environ['LANGSMITH_PROJECT']}")

### 2.3 Initialize the Models

Self-querying needs both: the **LLM** builds the structured query, the **embedding model** handles
the semantic half of the search.

In [ ]:
# ============================================================================
# MODEL INITIALIZATION: LLM (builds the query) + embeddings (semantic search)
# ============================================================================
llm = get_experientiallabs_llm()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"🤖 LLM:        {llm.model_name}")
print(f"🔢 Embeddings: {embeddings.model}")

---
## 📚 Part 3: A Corpus With Rich Metadata

Self-querying is only as good as your metadata. Each book carries seven attributes — `title`,
`author`, `year`, `genre`, `rating`, `language`, `country` — while `page_content` holds only a
short thematic description.

That split is deliberate and worth internalizing: **`page_content` is what gets embedded;
metadata is what gets filtered.** A question about *themes* is answered by the former, a question
about *ratings* by the latter.

In [ ]:
# ============================================================================
# CORPUS: Documents whose facts live in metadata, not in the text
# ============================================================================
docs = [
    Document(
        page_content="A complex, layered narrative exploring themes of identity and belonging",
        metadata={"title": "The Namesake", "author": "Jhumpa Lahiri", "year": 2003,
                  "genre": "Fiction", "rating": 4.5, "language": "English", "country": "USA"},
    ),
    Document(
        page_content="A luxurious, heartfelt novel with themes of love and loss set against a historical backdrop",
        metadata={"title": "The Nightingale", "author": "Kristin Hannah", "year": 2015,
                  "genre": "Historical Fiction", "rating": 4.8, "language": "English", "country": "France"},
    ),
    Document(
        page_content="A full-bodied epic with rich characters and a sprawling plot",
        metadata={"title": "War and Peace", "author": "Leo Tolstoy", "year": 1869,
                  "genre": "Historical Fiction", "rating": 4.7, "language": "Russian", "country": "Russia"},
    ),
    Document(
        page_content="An elegant, balanced narrative with intricate character development and subtle themes",
        metadata={"title": "Pride and Prejudice", "author": "Jane Austen", "year": 1813,
                  "genre": "Romance", "rating": 4.6, "language": "English", "country": "UK"},
    ),
    Document(
        page_content="A highly regarded novel with deep themes and a nuanced exploration of human nature",
        metadata={"title": "To Kill a Mockingbird", "author": "Harper Lee", "year": 1960,
                  "genre": "Fiction", "rating": 4.9, "language": "English", "country": "USA"},
    ),
    Document(
        page_content="A crisp, engaging story with vibrant characters and a compelling plot",
        metadata={"title": "The Alchemist", "author": "Paulo Coelho", "year": 1988,
                  "genre": "Adventure", "rating": 4.4, "language": "Portuguese", "country": "Brazil"},
    ),
    Document(
        page_content="A rich, complex narrative set in a dystopian future with strong thematic elements",
        metadata={"title": "1984", "author": "George Orwell", "year": 1949,
                  "genre": "Dystopian", "rating": 4.7, "language": "English", "country": "UK"},
    ),
    Document(
        page_content="An intense, gripping story with dark themes and intricate plot twists",
        metadata={"title": "Gone Girl", "author": "Gillian Flynn", "year": 2012,
                  "genre": "Thriller", "rating": 4.3, "language": "English", "country": "USA"},
    ),
    Document(
        page_content="An exotic, enchanting tale with rich descriptions and an intricate plot",
        metadata={"title": "One Hundred Years of Solitude", "author": "Gabriel García Márquez",
                  "year": 1967, "genre": "Magical Realism", "rating": 4.8,
                  "language": "Spanish", "country": "Colombia"},
    ),
]

print(f"📄 Corpus: {len(docs)} books")
print(f"📋 Metadata fields: {', '.join(docs[0].metadata.keys())}")

### 3.1 Index Into Chroma

In [ ]:
# ============================================================================
# VECTOR STORE: Embed page_content, keep metadata alongside for filtering
# ============================================================================
vectorstore = Chroma.from_documents(docs, embeddings)

print(f"✅ Indexed {len(docs)} books into Chroma")

---
## 🔍 Part 4: Baseline — Watch Plain Search Fail

Before using the self-querying retriever, confirm the problem is real. Ask plain similarity search
for books **rated above 4.5** and look at the ratings it actually returns.

In [ ]:
# ============================================================================
# BASELINE: Plain similarity search ignores the numeric constraint
# ============================================================================
baseline = vectorstore.similarity_search("a book with deep themes and a rating above 4.5", k=4)

print("🔍 Plain similarity search — 'rating above 4.5'\n")
for doc in baseline:
    rating = doc.metadata["rating"]
    flag = "✅" if rating > 4.5 else "❌"
    print(f"{flag} {rating}  {doc.metadata['title']}")

violations = [d for d in baseline if d.metadata["rating"] <= 4.5]
print(f"\n⚠️  {len(violations)} of {len(baseline)} results violate the stated constraint.")
print("   The phrase 'above 4.5' was embedded as text — never evaluated as arithmetic.")

---
## 🗂️ Part 5: Describe the Metadata Schema

The LLM cannot guess what fields exist or what they mean. `AttributeInfo` tells it — this schema is
injected into the query-construction prompt.

The `type` field matters more than it looks: declaring `rating` as `float` is what permits
comparisons like `gt`. Declared as `string`, the model would only be able to test equality.

> **Note**: `description` is a prompt, not documentation. *"The rating of the book (1-5 scale)"*
> tells the model the valid range, which helps it reject nonsense filters like `rating > 10`.

In [ ]:
# ============================================================================
# METADATA SCHEMA: What the LLM is allowed to filter on
# ============================================================================
metadata_field_info = [
    AttributeInfo(
        name="title",
        description="The title of the book",
        type="string or list[string]",
    ),
    AttributeInfo(
        name="author",
        description="The author of the book",
        type="string or list[string]",
    ),
    AttributeInfo(
        name="year",
        description="The year the book was published",
        type="integer",          # enables range comparisons on dates
    ),
    AttributeInfo(
        name="genre",
        description="The genre of the book",
        type="string or list[string]",
    ),
    AttributeInfo(
        name="rating",
        description="The rating of the book (1-5 scale)",
        type="float",            # enables gt / lt comparisons
    ),
    AttributeInfo(
        name="language",
        description="The language the book is written in",
        type="string",
    ),
    AttributeInfo(
        name="country",
        description="The country the author is from",
        type="string",
    ),
]

# Describes what page_content holds, so the LLM knows what the SEMANTIC half should target.
document_content_description = "Brief description of the book"

print(f"✅ Declared {len(metadata_field_info)} filterable attributes")

---
## 🔧 Part 6: Build the Self-Querying Retriever

> **⚠️ Why `structured_query_translator` is passed explicitly**
>
> Normally `from_llm()` infers the translator from the vector store type. That auto-detection is
> broken in this environment, for two independent reasons:
>
> 1. `_get_builtin_translator()` imports *every* supported vector store in a single statement,
>    including `DatabricksVectorSearch` — which no longer exists in `langchain-community` 0.4.2
>    (it moved to the `databricks-langchain` package). One missing name fails the whole import:
>    `ImportError: cannot import name 'DatabricksVectorSearch'`.
> 2. Even with that repaired, the builtin lookup table maps the **community** `Chroma` class. This
>    notebook uses `langchain_chroma.Chroma`, a different class, which would not match.
>
> Passing the translator explicitly short-circuits the lookup entirely
> (`langchain_classic/retrievers/self_query/base.py:373`), sidestepping both problems.

### 6.1 A Robustness Fix: Coercing Numeric Filter Values

The query constructor is an LLM, so its output is **not deterministic**. Asked the same question
four times, it produced the filter value `4.5` (float) twice, `"4.5"` (string) once, and no filter
at all once.

Chroma rejects the string form outright:

```
ValueError: Expected operand value to be an int or a float for operator $gt, got 4.5 in query.
```

The translator passes `comparison.value` straight through, so the string reaches Chroma unchanged.
A three-line subclass fixes it by casting values that *look* numeric, while leaving genuine strings
(like `country="USA"`) untouched.

> **Note**: this is worth remembering beyond this notebook. Any LLM-generated structured value
> should be validated or coerced before it reaches a system that type-checks it. The failure is
> intermittent, so it will pass your tests and break in production.

In [ ]:
# ============================================================================
# ROBUSTNESS: Coerce numeric filter values the LLM emitted as strings
# ============================================================================
class CoercingChromaTranslator(ChromaTranslator):
    """ChromaTranslator that casts numeric-looking string values to numbers.

    The query-construction LLM intermittently emits "4.5" instead of 4.5, which
    Chroma rejects. Genuine strings (country="USA") are left alone.
    """

    def visit_comparison(self, comparison: Comparison) -> dict:
        value = comparison.value
        if isinstance(value, str):
            try:
                value = float(value) if "." in value else int(value)
                comparison = Comparison(
                    comparator=comparison.comparator,
                    attribute=comparison.attribute,
                    value=value,
                )
            except ValueError:
                pass  # not numeric — a real string filter, leave as-is
        return super().visit_comparison(comparison)


print("✅ Coercing translator defined")

In [ ]:
# ============================================================================
# SELF-QUERY RETRIEVER: LLM + vector store + metadata schema
# ============================================================================
retriever = SelfQueryRetriever.from_llm(
    llm,                             # builds the structured query
    vectorstore,                     # runs the search
    document_content_description,    # what page_content contains
    metadata_field_info,             # what may be filtered on
    structured_query_translator=CoercingChromaTranslator(),  # see the note above
    verbose=True,
)

print("✅ Self-querying retriever ready")

---
## 🔬 Part 7: Look Inside — What the LLM Actually Produces

This is the cell that makes self-querying click. `retriever.query_constructor` is the LLM chain
that turns a question into a `StructuredQuery`; the translator then renders it as store syntax.
Calling them directly shows the decomposition instead of hiding it behind `.invoke()`.

In [ ]:
# ============================================================================
# INTROSPECTION: See the semantic query and the filter separately
# ============================================================================
probes = [
    "I want a book with deep themes and a rating above 4.5",
    "What books come from the USA?",
    "A historical fiction book published before 1900",
]

for q in probes:
    sq = retriever.query_constructor.invoke({"query": q})
    _, chroma_kwargs = retriever.structured_query_translator.visit_structured_query(sq)
    print(f"❓ {q}")
    print(f"   🔍 semantic query : {sq.query!r}")
    print(f"   📋 filter         : {sq.filter}")
    print(f"   🔧 chroma syntax  : {chroma_kwargs}\n")

Notice what happened to *"a rating above 4.5"*: the constraint was **removed** from the semantic
query (leaving just `'deep themes'`) and re-expressed as `{'rating': {'$gt': 4.5}}`. Each half now
goes to the tool that can actually evaluate it.

Also notice the second probe — *"What books come from the USA?"* — produces an essentially empty
semantic query. It is a **pure metadata lookup**: no meaning to match, only a fact to filter on.

---
## 🎯 Part 8: Example Queries

Each example exercises a different comparator. The `ask()` helper prints the **filter the LLM
produced** next to the results, so you can always tell whether a constraint was genuinely applied
or the retriever quietly fell back to plain semantic search.

In [ ]:
# ============================================================================
# HELPER: One invocation, showing the filter that actually produced the results
# ============================================================================
# This mirrors SelfQueryRetriever._get_relevant_documents: build the structured
# query ONCE, then run the search from it. Calling retriever.invoke() separately
# would issue a SECOND LLM call, so the filter displayed could differ from the
# one that produced the results — the query constructor is non-deterministic.
def ask(retriever, query, fields=("rating", "year", "genre", "country")):
    structured = retriever.query_constructor.invoke({"query": query})
    new_query, search_kwargs = retriever._prepare_query(query, structured)
    results = retriever.vectorstore.search(new_query, retriever.search_type, **search_kwargs)

    print(f"❓ {query}")
    print(f"   🔍 semantic query : {structured.query!r}")
    print(f"   📋 filter         : {structured.filter}")
    if structured.limit is not None:
        print(f"   🔢 limit          : {structured.limit}")
    print(f"   📊 {len(results)} result(s):")
    for doc in results:
        meta = "  ".join(f"{f}={doc.metadata[f]}" for f in fields)
        print(f"      📄 {doc.metadata["title"]:<32} {meta}")
    if structured.filter is None:
        print("   ⚠️  No filter generated — these are pure semantic matches.")
    print()
    return results


print("✅ Helper ready")

### 8.1 Semantic + Category Filter
Combines meaning (*"highly rated"*) with a `genre` match.

In [ ]:
# ============================================================================
# QUERY 1: Genre filter
# ============================================================================
ask(retriever, "What are some highly rated historical fiction books")

### 8.2 Numeric Comparison (`$gt`)
The query that defeated plain search in Part 4.

In [ ]:
# ============================================================================
# QUERY 2: Numeric comparison — compare against the Part 4 baseline
# ============================================================================
ask(retriever, "I want a book with deep themes and a rating above 4.5")

### 8.3 Purely Semantic
No constraint here, so no filter is generated — this degrades gracefully to ordinary
similarity search.

In [ ]:
# ============================================================================
# QUERY 3: No filter — plain semantic retrieval
# ============================================================================
ask(retriever, "I want a book with complex characters and a gripping plot")

### 8.4 Pure Metadata Lookup (`$eq`)
Almost no semantic content — the whole question is a fact lookup.

In [ ]:
# ============================================================================
# QUERY 4: Equality filter
# ============================================================================
ask(retriever, "What books come from the USA?")

### 8.5 Composite Filter (range + semantics)
Two comparators combined with `AND`, plus a semantic component.

In [ ]:
# ============================================================================
# QUERY 5: Range filter (year > 2003 AND year < 2015) + semantic match
# ============================================================================
ask(
    retriever,
    "What's a book published after 2003 but before 2015 with deep themes and a high rating",
)

---
## 🔢 Part 9: Letting the Query Control `k`

With `enable_limit=True`, the LLM may also extract a **result count** from the question — *"two
books"* becomes `limit=2`. Without it, that phrase is silently ignored and you get the default `k`.

In [ ]:
# ============================================================================
# ENABLE_LIMIT: Let the question specify how many results to return
# ============================================================================
retriever_with_limit = SelfQueryRetriever.from_llm(
    llm,
    vectorstore,
    document_content_description,
    metadata_field_info,
    structured_query_translator=CoercingChromaTranslator(),
    enable_limit=True,   # allows the LLM to set a result limit
    verbose=True,
)

print("✅ Retriever with limit extraction ready")

### 9.1 Limit + Numeric Filter

In [ ]:
# ============================================================================
# QUERY 6: 'two books' -> limit=2, plus a rating filter
# ============================================================================
ask(retriever_with_limit, "What are two books that have a rating above 4.8")

### 9.2 Limit + Set Membership (`OR`)

In [ ]:
# ============================================================================
# QUERY 7: 'two books' -> limit=2, plus country in (USA, UK)
# ============================================================================
ask(retriever_with_limit, "What are two books that come from USA or UK")

### 9.3 Confirm the Limit Is Real

Compare the same question through both retrievers. Only the `enable_limit=True` one honors
*"two books"*.

In [ ]:
# ============================================================================
# COMPARISON: Does 'two books' actually constrain the result count?
# ============================================================================
# The limit shows up in the StructuredQuery, which is the reliable place to
# check. Result counts alone can coincide when few documents match the filter.
q = "What are two books that come from USA or UK"

sq_without = retriever.query_constructor.invoke({"query": q})
sq_with = retriever_with_limit.query_constructor.invoke({"query": q})

print(f"❌ enable_limit=False -> limit={sq_without.limit!r}, {len(retriever.invoke(q))} results")
print(f"✅ enable_limit=True  -> limit={sq_with.limit!r}, {len(retriever_with_limit.invoke(q))} results")

---
## 📝 Summary

### 1. The Problem
- Embeddings encode **meaning**, not **facts**. The phrase *"rating above 4.5"* has a vector, but
  that vector cannot perform a comparison — Part 4 showed plain search returning 4.3-rated books.

### 2. How Self-Querying Works
- An LLM splits one question into a **semantic query** (for the embeddings) and a **structured
  filter** (for the metadata index), then both are applied together.
- Part 7 made this visible: *"deep themes and a rating above 4.5"* became `'deep themes'` plus
  `{'rating': {'$gt': 4.5}}`.

### 3. The Metadata Schema Is the Contract
- `AttributeInfo` is what the LLM sees. A field you don't declare cannot be filtered on.
- **`type` decides which comparators are possible** — `float`/`integer` unlock `gt`/`lt`;
  `string` effectively limits you to equality.
- `description` is prompt text, not documentation. Stating the valid range guards against
  nonsensical filters.

### 4. Translators
- The query constructor emits a **store-agnostic** filter; a translator renders it into a specific
  dialect (Chroma's `$gt`, Pinecone's syntax, a SQL `WHERE`, …).
- Auto-detection is fragile. Passing `structured_query_translator` explicitly is both a workaround
  for the broken `langchain-community` import chain **and** the more predictable habit.

### 5. Costs and Caveats
- Every query now costs an **LLM call before retrieval** — slower and more expensive than plain search.
- A malformed or hallucinated filter yields **zero results**, which looks like "nothing matched"
  rather than an error. Inspect `query_constructor` output (Part 7) when debugging empty results.
- Metadata must be clean and consistent; `"USA"` vs `"United States"` will silently break equality filters.

### 6. `enable_limit`
- Off by default. Enable it when users naturally say *"show me three…"*, and expect the count to be
  ignored otherwise.

### Next Steps
- Inspect these runs in LangSmith under the **Self-Querying-Retrieval** project to see the
  query-construction call that precedes each retrieval.
- Compare with the sibling routing techniques in this folder: `a. Routing_LLM_Classifier`
  (picks a data source) and `b. Semantic_Routing` (picks a prompt). Self-querying instead builds a
  **filter** over one source.